# Movie Recommendation System - ML Project
### Dataset: MovieLens (10,329 Movies | 105,339 Ratings | 668 Users)
---
**Techniques Used:**
- **Content-Based Filtering** — Genre similarity (TF-IDF + Cosine Similarity)
- **Collaborative Filtering** — User-Item Matrix (SVD / KNN)
- **Hybrid Recommendation** — Combines both approaches

**Files Required:** `movies.csv`, `ratings.csv`

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully!")

## Step 2: Load Dataset

In [ ]:
movies  = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("=== MOVIES ===")
print(f"Shape : {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print(movies.head())

print("\n=== RATINGS ===")
print(f"Shape : {ratings.shape[0]:,} rows × {ratings.shape[1]} columns")
print(ratings.head())

print(f"\n Unique Users  : {ratings['userId'].nunique():,}")
print(f" Unique Movies : {ratings['movieId'].nunique():,}")
print(f" Rating Range  : {ratings['rating'].min()} – {ratings['rating'].max()}")

##  Step 3: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Movie Recommendation — EDA Overview', fontsize=16, fontweight='bold')

# 1. Rating Distribution
axes[0,0].hist(ratings['rating'], bins=10, color='#4F46E5',
               edgecolor='white', alpha=0.85, rwidth=0.85)
axes[0,0].set_title('Rating Distribution', fontweight='bold')
axes[0,0].set_xlabel('Rating')
axes[0,0].set_ylabel('Count')
axes[0,0].axvline(ratings['rating'].mean(), color='red', linestyle='--',
                  linewidth=2, label=f"Mean: {ratings['rating'].mean():.2f}")
axes[0,0].legend()

# 2. Top 15 Most Rated Movies
top_movies = (ratings.groupby('movieId')['rating']
              .count()
              .reset_index()
              .rename(columns={'rating': 'count'})
              .merge(movies[['movieId','title']], on='movieId')
              .sort_values('count', ascending=False)
              .head(15))
top_movies['short_title'] = top_movies['title'].str[:25]
axes[0,1].barh(top_movies['short_title'], top_movies['count'],
               color='#7C3AED', edgecolor='white', alpha=0.85)
axes[0,1].set_title('Top 15 Most Rated Movies', fontweight='bold')
axes[0,1].set_xlabel('Number of Ratings')
axes[0,1].invert_yaxis()

# 3. Top 10 Genres
all_genres = movies['genres'].str.split('|').explode()
genre_counts = all_genres.value_counts().drop('(no genres listed)', errors='ignore').head(10)
axes[1,0].bar(genre_counts.index, genre_counts.values,
              color='#06B6D4', edgecolor='white', alpha=0.85)
axes[1,0].set_title('Top 10 Genres', fontweight='bold')
axes[1,0].set_xlabel('Genre')
axes[1,0].set_ylabel('Number of Movies')
axes[1,0].tick_params(axis='x', rotation=35)

# 4. Ratings per User Distribution
user_counts = ratings.groupby('userId')['rating'].count()
axes[1,1].hist(user_counts, bins=30, color='#10B981',
               edgecolor='white', alpha=0.85)
axes[1,1].set_title('Ratings per User', fontweight='bold')
axes[1,1].set_xlabel('Number of Ratings')
axes[1,1].set_ylabel('Number of Users')

for ax in axes.flat:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 Highest Rated Movies (min 50 ratings)
movie_stats = (ratings.groupby('movieId')['rating']
               .agg(['mean','count'])
               .reset_index()
               .rename(columns={'mean':'avg_rating','count':'num_ratings'}))
movie_stats = movie_stats[movie_stats['num_ratings'] >= 50]
top_rated = (movie_stats.sort_values('avg_rating', ascending=False)
             .head(15)
             .merge(movies[['movieId','title']], on='movieId'))

plt.figure(figsize=(11, 6))
bars = plt.barh(top_rated['title'].str[:35].iloc[::-1],
                top_rated['avg_rating'].iloc[::-1],
                color='#F59E0B', edgecolor='white', alpha=0.9)
plt.title('Top 15 Highest Rated Movies (min 50 ratings)', fontweight='bold', fontsize=13)
plt.xlabel('Average Rating')
plt.xlim(3.5, 5.2)
for bar, val in zip(bars, top_rated['avg_rating'].iloc[::-1]):
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

##  Step 4: Content-Based Filtering

In [ ]:
# TF-IDF on genres
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genres_clean'] = movies['genres_clean'].replace('(no genres listed)', '')

tfidf     = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
cosine_sim   = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Index mapping
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print(f" TF-IDF Matrix Shape : {tfidf_matrix.shape}")
print(f" Cosine Sim Matrix   : {cosine_sim.shape}")
print(f"\n Sample genres_clean:")
print(movies[['title','genres_clean']].head(5).to_string(index=False))

In [ ]:
def content_based_recommend(title, n=10):
    """Recommend movies based on genre similarity."""
    # Find closest title match
    matches = [t for t in indices.index if title.lower() in t.lower()]
    if not matches:
        print(f"❌ '{title}' not found. Try a different name.")
        return pd.DataFrame()

    matched_title = matches[0]
    idx    = indices[matched_title]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    movie_indices = [i[0] for i in scores]

    rec = movies.iloc[movie_indices][['title','genres']].copy()
    rec['similarity_score'] = [round(i[1], 4) for i in scores]
    rec = rec.reset_index(drop=True)
    rec.index += 1

    print(f"\n🎬 Because you liked: '{matched_title}'")
    print(f" Top {n} Content-Based Recommendations:\n")
    return rec

# ── Test it
content_based_recommend("Toy Story", n=10)

##  Step 5: Collaborative Filtering

In [ ]:
# Build User-Item Matrix (top 2000 movies for memory efficiency)
top_movie_ids = (ratings.groupby('movieId')['rating']
                 .count()
                 .sort_values(ascending=False)
                 .head(2000)
                 .index)

ratings_filtered = ratings[ratings['movieId'].isin(top_movie_ids)]
user_item_matrix = (ratings_filtered
                    .pivot_table(index='userId', columns='movieId', values='rating')
                    .fillna(0))

print(f" User-Item Matrix Shape: {user_item_matrix.shape}")
print(f"   Users  : {user_item_matrix.shape[0]}")
print(f"   Movies : {user_item_matrix.shape[1]}")
sparsity = 1 - (ratings_filtered.shape[0] / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))
print(f"   Sparsity : {sparsity:.2%}")

In [ ]:
# SVD — Matrix Factorization
svd   = TruncatedSVD(n_components=50, random_state=42)
matrix_svd = svd.fit_transform(user_item_matrix)

explained_variance = svd.explained_variance_ratio_.cumsum()

plt.figure(figsize=(8, 4))
plt.plot(range(1, 51), explained_variance, color='#4F46E5', linewidth=2.5, marker='o', markersize=3)
plt.axhline(0.8, color='red', linestyle='--', linewidth=1.5, label='80% Variance')
plt.title('SVD — Cumulative Explained Variance', fontweight='bold', fontsize=13)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

print(f" SVD complete! 50 components explain {explained_variance[-1]:.2%} of variance")

In [ ]:
# KNN on SVD-reduced matrix
knn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=11)
knn_model.fit(user_item_matrix.T)  # Movie-based (transpose)

# Movie index mapping within filtered matrix
movie_mapper  = {movie: i for i, movie in enumerate(user_item_matrix.columns)}
movie_inv_map = {i: movie for movie, i in movie_mapper.items()}

def collaborative_recommend(title, n=10):
    """Recommend movies using KNN Collaborative Filtering."""
    matches = movies[movies['title'].str.contains(title, case=False, na=False)]
    if matches.empty:
        print(f"❌ '{title}' not found.")
        return pd.DataFrame()

    movie_id = matches.iloc[0]['movieId']
    if movie_id not in movie_mapper:
        print(f"'{title}' is in dataset but has too few ratings for CF. Try Content-Based.")
        return pd.DataFrame()

    idx = movie_mapper[movie_id]
    distances, indices_knn = knn_model.kneighbors(
        user_item_matrix.T.iloc[idx].values.reshape(1, -1), n_neighbors=n+1
    )

    results = []
    for i, dist in zip(indices_knn.flatten()[1:], distances.flatten()[1:]):
        mid = movie_inv_map[i]
        title_found = movies[movies['movieId'] == mid]['title'].values
        genre_found = movies[movies['movieId'] == mid]['genres'].values
        if len(title_found):
            results.append({
                'title': title_found[0],
                'genres': genre_found[0],
                'similarity_score': round(1 - dist, 4)
            })

    rec = pd.DataFrame(results).reset_index(drop=True)
    rec.index += 1

    print(f"\n Because you liked: '{matches.iloc[0]['title']}'")
    print(f" Top {n} Collaborative Filtering Recommendations:\n")
    return rec

# ── Test it
collaborative_recommend("Toy Story", n=10)

##  Step 6: Hybrid Recommendation System


In [ ]:
def hybrid_recommend(title, n=10, cb_weight=0.4, cf_weight=0.6):
    """
    Hybrid: Content-Based + Collaborative Filtering combined.
    cb_weight: weight for content-based score (default 0.4)
    cf_weight: weight for collaborative score  (default 0.6)
    """
    # --- Content-Based ---
    matches = [t for t in indices.index if title.lower() in t.lower()]
    if not matches:
        print(f"❌ '{title}' not found.")
        return pd.DataFrame()

    matched_title = matches[0]
    idx_cb = indices[matched_title]
    cb_scores = dict(enumerate(cosine_sim[idx_cb]))

    # --- Collaborative ---
    movie_row = movies[movies['title'] == matched_title]
    if movie_row.empty or movie_row.iloc[0]['movieId'] not in movie_mapper:
        print(f"No CF data. Falling back to Content-Based only.")
        return content_based_recommend(title, n)

    movie_id = movie_row.iloc[0]['movieId']
    idx_cf = movie_mapper[movie_id]
    distances, cf_indices = knn_model.kneighbors(
        user_item_matrix.T.iloc[idx_cf].values.reshape(1, -1), n_neighbors=50
    )
    cf_scores = {movie_inv_map[i]: 1 - d
                 for i, d in zip(cf_indices.flatten()[1:], distances.flatten()[1:])}

    # --- Combine Scores ---
    hybrid_scores = []
    for i, row in movies.iterrows():
        if row['title'] == matched_title:
            continue
        cb = cb_scores.get(i, 0)
        cf = cf_scores.get(row['movieId'], 0)
        combined = (cb_weight * cb) + (cf_weight * cf)
        hybrid_scores.append({
            'title': row['title'],
            'genres': row['genres'],
            'cb_score': round(cb, 4),
            'cf_score': round(cf, 4),
            'hybrid_score': round(combined, 4)
        })

    rec = (pd.DataFrame(hybrid_scores)
           .sort_values('hybrid_score', ascending=False)
           .head(n)
           .reset_index(drop=True))
    rec.index += 1

    print(f"\n Because you liked: '{matched_title}'")
    print(f" Top {n} Hybrid Recommendations (CB:{cb_weight} + CF:{cf_weight}):\n")
    return rec

# ── Test it
hybrid_recommend("Toy Story", n=10)

## Step 7: Model Evaluation (RMSE)

In [ ]:
# Evaluate SVD-based rating prediction using RMSE
train_data, test_data = train_test_split(ratings_filtered, test_size=0.2, random_state=42)

# Rebuild matrix on train only
train_matrix = (train_data.pivot_table(index='userId', columns='movieId', values='rating')
                .fillna(0))

# Align columns
common_cols = user_item_matrix.columns.intersection(train_matrix.columns)
train_matrix = train_matrix.reindex(columns=user_item_matrix.columns, fill_value=0)

svd_eval = TruncatedSVD(n_components=50, random_state=42)
U = svd_eval.fit_transform(train_matrix)
Vt = svd_eval.components_
predicted_matrix = np.dot(U, Vt)
predicted_df = pd.DataFrame(predicted_matrix,
                             index=train_matrix.index,
                             columns=train_matrix.columns)

# Calculate RMSE on test set
actuals, preds = [], []
for _, row in test_data.iterrows():
    uid, mid, actual = row['userId'], row['movieId'], row['rating']
    if uid in predicted_df.index and mid in predicted_df.columns:
        pred = predicted_df.loc[uid, mid]
        actuals.append(actual)
        preds.append(pred)

rmse = np.sqrt(mean_squared_error(actuals, preds))
print(f"SVD Collaborative Filtering RMSE : {rmse:.4f}")
print(f"   (Lower is better | Scale: 0.5 – 5.0)")
print(f"   Evaluated on {len(actuals):,} test ratings")

## Step 8: Recommendation Visualization

In [ ]:
# Visualize Content-Based vs Hybrid scores side by side
cb_recs     = content_based_recommend("Toy Story", n=8)
hybrid_recs = hybrid_recommend("Toy Story", n=8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Recommendations for "Toy Story"', fontsize=15, fontweight='bold')

# Content-Based
if not cb_recs.empty:
    axes[0].barh(cb_recs['title'].str[:30].iloc[::-1],
                 cb_recs['similarity_score'].iloc[::-1],
                 color='#4F46E5', edgecolor='white', alpha=0.85)
    axes[0].set_title('Content-Based Filtering', fontweight='bold')
    axes[0].set_xlabel('Cosine Similarity Score')

# Hybrid
if not hybrid_recs.empty:
    axes[1].barh(hybrid_recs['title'].str[:30].iloc[::-1],
                 hybrid_recs['hybrid_score'].iloc[::-1],
                 color='#10B981', edgecolor='white', alpha=0.85)
    axes[1].set_title('Hybrid Recommendation', fontweight='bold')
    axes[1].set_xlabel('Hybrid Score')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## Step 9: Try Your Own Movie!

In [ ]:
# Change the movie name here and run!
my_movie = "Jumanji"

print("=" * 55)
print(f"  Recommendations for: {my_movie}")
print("=" * 55)

print("\n1. CONTENT-BASED:")
display(content_based_recommend(my_movie, n=5))

print("\n2. COLLABORATIVE FILTERING:")
display(collaborative_recommend(my_movie, n=5))

print("\n3. HYBRID:")
display(hybrid_recommend(my_movie, n=5))

## Step 10: Search Movies by Genre

In [ ]:
def search_by_genre(genre, min_ratings=30, n=10):
    """Find top-rated movies by genre."""
    genre_movies = movies[movies['genres'].str.contains(genre, case=False, na=False)]
    stats = (ratings.groupby('movieId')['rating']
             .agg(['mean','count'])
             .reset_index()
             .rename(columns={'mean':'avg_rating','count':'num_ratings'}))
    stats = stats[stats['num_ratings'] >= min_ratings]
    result = (genre_movies.merge(stats, on='movieId')
              .sort_values('avg_rating', ascending=False)
              .head(n)[['title','genres','avg_rating','num_ratings']]
              .reset_index(drop=True))
    result.index += 1
    result['avg_rating'] = result['avg_rating'].round(2)
    print(f"\n🎭 Top {n} '{genre}' Movies (min {min_ratings} ratings):\n")
    return result

# Try different genres: Action, Comedy, Drama, Horror, Sci-Fi, Romance
search_by_genre("Action", n=10)